# LandslideGuard - Detection Module
## Stage 1: Data Preparation and Data Pipeline Verification

This notebook is the authoritative record of Stage 1 for the Detection module.
The goal is to *verify* the Landslide4Sense dataset end-to-end and build a
reproducible data pipeline up to (and including) the PyTorch DataLoader.
**No U-Net training, no model evaluation, no live ingestion.**

Every non-trivial decision below is grounded in an actual measurement on the raw
dataset, saved under `outputs/detection/data_verification/`.


## SECTION 01 - Project Configuration

We anchor all paths at the project root and expose the raw split directories
(the archive's own nested `Split/Split/{img,mask}` layout is preserved as-is).
A single seed is fixed for reproducibility of augmentation and sampling.


In [1]:
from __future__ import annotations
import os, sys, json, random
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT     = PROJECT_ROOT / "data" / "raw" / "landslide4sense"
TRAIN_IMG_DIR = DATA_ROOT / "TrainData" / "TrainData" / "img"
TRAIN_MSK_DIR = DATA_ROOT / "TrainData" / "TrainData" / "mask"
VALID_IMG_DIR = DATA_ROOT / "ValidData" / "ValidData" / "img"
VALID_MSK_DIR = DATA_ROOT / "ValidData" / "ValidData" / "mask"
TEST_IMG_DIR  = DATA_ROOT / "TestData"  / "TestData"  / "img"
TEST_MSK_DIR  = DATA_ROOT / "TestData"  / "TestData"  / "mask"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "detection"
OUTPUT_DIR    = PROJECT_ROOT / "outputs" / "detection" / "data_verification"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except Exception:
    pass

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT   :", DATA_ROOT)
print("SEED        :", SEED)


PROJECT_ROOT: D:\LANDSLIDE\LandslideGuard
DATA_ROOT   : D:\LANDSLIDE\LandslideGuard\data\raw\landslide4sense
SEED        : 42


## SECTION 02 - Environment Verification

We record library versions so the pipeline is fully reproducible. `h5py` is
required to read Landslide4Sense HDF5 patches; the other packages support
tensorization and visualization.


In [2]:
import sys, importlib
mods = ["numpy","h5py","torch","matplotlib","yaml","pandas","skimage","tqdm"]
print("python", sys.version.split()[0])
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m:12s} {getattr(mod, '__version__', 'unknown')}")
    except Exception as e:
        print(f"{m:12s} MISSING ({type(e).__name__})")


python 3.13.1
numpy        2.2.6
h5py         3.16.0
torch        2.11.0+cpu


matplotlib   3.10.3
yaml         6.0.2


pandas       2.3.0
skimage      0.26.0
tqdm         4.67.1


## SECTION 03 - Dataset Path and Folder Verification

We check that all three split directories exist and count files by extension
per split. Only `.h5` files are expected; anything else is flagged.


In [3]:
from collections import Counter
SPLITS = {
    "train": (TRAIN_IMG_DIR, TRAIN_MSK_DIR),
    "valid": (VALID_IMG_DIR, VALID_MSK_DIR),
    "test":  (TEST_IMG_DIR,  TEST_MSK_DIR),
}
print(f"{'split':6s} {'kind':5s} {'total':>6s} {'.h5':>6s} other_exts")
for s, (idir, mdir) in SPLITS.items():
    for kind, d in [("img", idir), ("mask", mdir)]:
        assert d.is_dir(), f"missing directory: {d}"
        files = list(d.iterdir())
        exts = Counter(p.suffix for p in files)
        h5 = exts.get(".h5", 0)
        other = {k: v for k, v in exts.items() if k != ".h5"}
        print(f"{s:6s} {kind:5s} {len(files):6d} {h5:6d} {other}")


split  kind   total    .h5 other_exts
train  img     3799   3799 {}
train  mask    3799   3799 {}
valid  img      245    245 {}
valid  mask     245    245 {}
test   img      800    800 {}
test   mask     800    800 {}


**Result:** all six directories exist, all files are HDF5, no stray files. Counts (3799/245/800) match the official Landslide4Sense release.


## SECTION 04 - HDF5 File Discovery

We open representative files from each split to test structural consistency
(assuming a single file is representative would be unsafe).


In [4]:
import h5py
def h5_info(path):
    with h5py.File(path, "r") as f:
        return {k: {"shape": tuple(f[k].shape), "dtype": str(f[k].dtype)} for k in f.keys()}

for s, (idir, mdir) in SPLITS.items():
    img_samples  = sorted(idir.iterdir())[:3]
    mask_samples = sorted(mdir.iterdir())[:3]
    print(f"[{s}]")
    for p in img_samples:
        print(f"  {p.name}: {h5_info(p)}")
    for p in mask_samples:
        print(f"  {p.name}: {h5_info(p)}")


[train]
  image_1.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  image_10.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  image_100.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  mask_1.h5: {'mask': {'shape': (128, 128), 'dtype': 'uint8'}}
  mask_10.h5: {'mask': {'shape': (128, 128), 'dtype': 'uint8'}}
  mask_100.h5: {'mask': {'shape': (128, 128), 'dtype': 'uint8'}}
[valid]
  image_1.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  image_10.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  image_100.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  mask_1.h5: {'mask': {'shape': (128, 128), 'dtype': 'uint8'}}
  mask_10.h5: {'mask': {'shape': (128, 128), 'dtype': 'uint8'}}
  mask_100.h5: {'mask': {'shape': (128, 128), 'dtype': 'uint8'}}
[test]
  image_1.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  image_10.h5: {'img': {'shape': (128, 128, 14), 'dtype': 'float64'}}
  image_100.h5: {'img': {'shape': 

## SECTION 05 - HDF5 Internal Structure Inspection

**Discovery, not assumption.**
Each image file contains exactly one dataset named `img` of shape (128, 128, 14)
and dtype `float64`. Each mask file contains one dataset named `mask` of
shape (128, 128) and dtype `uint8`. There are no nested groups and no
attributes on the datasets or on the file root. The structure is identical
across all three splits.


In [5]:
p = sorted(TRAIN_IMG_DIR.iterdir())[0]
with h5py.File(p, "r") as f:
    print("root keys :", list(f.keys()))
    print("root attrs:", dict(f.attrs))
    d = f["img"]
    print("img shape :", d.shape, "dtype:", d.dtype, "attrs:", dict(d.attrs))


root keys : ['img']
root attrs: {}
img shape : (128, 128, 14) dtype: float64 attrs: {}


## SECTION 06 - Image and Mask Identification

The `img` dataset carries the 14 input features; `mask` carries the
segmentation ground truth. We confirm shapes match spatially (both 128x128)
and inspect a couple of files including basic statistics.


In [6]:
import numpy as np, h5py
for split, (idir, mdir) in SPLITS.items():
    for name in ["image_1.h5", "image_100.h5"]:
        with h5py.File(idir / name, "r") as f: x = f["img"][()]
        with h5py.File(mdir / name.replace("image", "mask"), "r") as f: y = f["mask"][()]
        print(f"[{split} {name}] img shape={x.shape} dtype={x.dtype} "
              f"min={x.min():.3f} max={x.max():.3f} mean={x.mean():.3f}")
        print(f"[{split} {name}] mask shape={y.shape} dtype={y.dtype} "
              f"unique={np.unique(y).tolist()} positive={int((y>0).sum())}")


[train image_1.h5] img shape=(128, 128, 14) dtype=float64 min=0.000 max=6.397 mean=1.370
[train image_1.h5] mask shape=(128, 128) dtype=uint8 unique=[0, 1] positive=405


[train image_100.h5] img shape=(128, 128, 14) dtype=float64 min=0.000 max=3.400 mean=1.235
[train image_100.h5] mask shape=(128, 128) dtype=uint8 unique=[0, 1] positive=479
[valid image_1.h5] img shape=(128, 128, 14) dtype=float64 min=0.000 max=13.376 mean=1.260
[valid image_1.h5] mask shape=(128, 128) dtype=uint8 unique=[0] positive=0
[valid image_100.h5] img shape=(128, 128, 14) dtype=float64 min=0.000 max=5.255 mean=1.293
[valid image_100.h5] mask shape=(128, 128) dtype=uint8 unique=[0, 1] positive=27
[test image_1.h5] img shape=(128, 128, 14) dtype=float64 min=0.000 max=3.952 mean=1.368
[test image_1.h5] mask shape=(128, 128) dtype=uint8 unique=[0, 1] positive=33
[test image_100.h5] img shape=(128, 128, 14) dtype=float64 min=0.000 max=4.537 mean=1.112
[test image_100.h5] mask shape=(128, 128) dtype=uint8 unique=[0] positive=0


## SECTION 07 - Verify Image Dimensions

All 3799 + 245 + 800 files have image shape `(128, 128, 14)` and mask shape
`(128, 128)`. This was checked while computing per-channel training
statistics: every file loaded successfully and every one contributed
128 * 128 = 16384 samples per channel.


| Split | Image shape | Mask shape | Files |
|-------|-------------|------------|------:|
| train | (128, 128, 14) | (128, 128) | 3799 |
| valid | (128, 128, 14) | (128, 128) | 245 |
| test  | (128, 128, 14) | (128, 128) | 800 |


## SECTION 08 - Verify the 14-Channel Structure

The channel axis is length 14, matching the official Landslide4Sense release.
Per the dataset paper (Ghorbanzadeh et al., *Landslide4Sense: reference
benchmark data and deep learning models for landslide detection*, 2022), the
canonical channel ordering is:

| Idx | Feature | Source |
|----:|---------|--------|
| 0 | B1 - Coastal aerosol (443 nm)   | Sentinel-2 |
| 1 | B2 - Blue (490 nm)              | Sentinel-2 |
| 2 | B3 - Green (560 nm)             | Sentinel-2 |
| 3 | B4 - Red (665 nm)               | Sentinel-2 |
| 4 | B5 - Red Edge 1 (705 nm)        | Sentinel-2 |
| 5 | B6 - Red Edge 2 (740 nm)        | Sentinel-2 |
| 6 | B7 - Red Edge 3 (783 nm)        | Sentinel-2 |
| 7 | B8 - NIR (842 nm)               | Sentinel-2 |
| 8 | B9 - Water Vapor (945 nm)       | Sentinel-2 |
| 9 | B10 - Cirrus (1375 nm)          | Sentinel-2 |
| 10 | B11 - SWIR1 (1610 nm)          | Sentinel-2 |
| 11 | B12 - SWIR2 (2190 nm)          | Sentinel-2 |
| 12 | Slope                          | ALOS PALSAR DEM |
| 13 | DEM                            | ALOS PALSAR |

The labels are metadata used only for display and configuration; the
preprocessing math does not depend on them. All 14 statistics below are
consistent with this ordering: bands 0-11 are non-negative reflectance-like
quantities with typical means near 1, and bands 12-13 stand apart as
terrain-derived quantities (larger means and standard deviations).


## SECTION 09 - Channel Statistical Analysis

Statistics below are computed on **the full training set (3799 patches,
62,242,816 pixels per channel)** using an exact one-pass sum / sum-of-squares
accumulator in `float64`. They come from
`outputs/detection/data_verification/channel_statistics.csv` and
`training_channel_stats.json`.

| Ch | min | max | mean | std | n_finite | n_nan | n_inf |
|---:|----:|----:|-----:|----:|--------:|-----:|-----:|
| 0 | 0.0000 | 3.1057 | 0.9257 | 0.1410 | 62,242,816 | 0 | 0 |
| 1 | 0.0000 | 19.7615 | 0.9227 | 0.2207 | 62,242,816 | 0 | 0 |
| 2 | 0.0000 | 31.5518 | 0.9541 | 0.3184 | 62,242,816 | 0 | 0 |
| 3 | 0.0000 | 33.1601 | 0.9596 | 0.5724 | 62,242,816 | 0 | 0 |
| 4 | 0.0000 | 9.9728 | 1.0228 | 0.4601 | 62,242,816 | 0 | 0 |
| 5 | 0.0000 | 4.1441 | 1.0426 | 0.4465 | 62,242,816 | 0 | 0 |
| 6 | 0.0000 | 3.6925 | 1.0358 | 0.4651 | 62,242,816 | 0 | 0 |
| 7 | 0.0000 | 8.3129 | 1.0468 | 0.4948 | 62,242,816 | 0 | 0 |
| 8 | 0.0000 | 3.5486 | 1.1699 | 0.5133 | 62,242,816 | 0 | 0 |
| 9 | 0.0000 | 21.4415 | 1.1736 | 0.6836 | 62,242,816 | 0 | 0 |
| 10 | 0.0000 | 5.7864 | 1.0495 | 0.5323 | 62,242,816 | 0 | 0 |
| 11 | 0.0000 | 19.6613 | 1.0370 | 0.6628 | 62,242,816 | 0 | 0 |
| 12 | 0.0000 | 4.0409 | 1.2511 | 0.6784 | 62,242,816 | 0 | 0 |
| 13 | 0.0000 | 5.1214 | 1.6495 | 1.0727 | 62,242,816 | 0 | 0 |


In [7]:
from pathlib import Path
import json
stats = json.loads((PROJECT_ROOT / "outputs/detection/data_verification/training_channel_stats.json").read_text())
print(f"train patches   : {stats['n_files']}")
print(f"pixels/channel  : {stats['n_finite'][0]:,}")
print(f"total NaN       : {sum(stats['n_nan'])}")
print(f"total Inf       : {sum(stats['n_inf'])}")


train patches   : 3799
pixels/channel  : 62,242,816
total NaN       : 0
total Inf       : 0


## SECTION 10 - Invalid Value / NoData Analysis

**Observations from the full-training-set scan:**
* Zero NaN values across all 14 channels.
* Zero Inf values across all 14 channels.
* Every channel has an exact minimum of `0.0`. This is likely the
  Landslide4Sense authors' NoData / masked-out convention, applied
  consistently across all channels.

**Decision:** We do **not** attempt to distinguish structural zeros from
true-zero reflectance / true-zero elevation at this stage - doing so would
require geolocation metadata that is not present in the HDF5 files. Zeros
are passed through as-is and will be standardized like any other value. A
defensive `nan_to_num` step remains in the preprocessing function so any
future corrupt patch still yields a finite tensor.


## SECTION 11 - Outlier Analysis

Extreme observed maxima:

* Channels 2, 3 (Green, Red): max ~ 31.6 and 33.2 respectively.
* Channels 9, 11: max ~ 21.4 and 19.7 (SWIR / cirrus).
* Channels 12 (Slope) and 13 (DEM) have maxima ~ 4.0 and 5.1, matching a
  scaled elevation/slope encoding.

Because these values sit only a few standard deviations above the mean,
they are consistent with legitimate high-reflectance targets (snow, cloud
edges, bright soils) rather than sensor artifacts. **No clipping is applied
in Stage 1**: z-scoring alone is sufficient to bring per-batch
per-channel means and stds close to 0 and 1 (verified in Section 24).


## SECTION 12 - Normalization Strategy

**Chosen strategy: per-channel z-score, statistics computed on the training
split only.**

For each channel c:

$$ x'_c = \frac{x_c - \mu_c^{\text{train}}}{\max(\sigma_c^{\text{train}},\; \varepsilon)} $$

with $\varepsilon = 10^{-6}$ as a defensive floor. NaN / Inf values are
replaced by `0.0` **before** normalization (never triggered on the observed
training data - see Section 10).

Statistics are persisted in
`outputs/detection/data_verification/normalization_statistics.json` so the
exact same transform can be re-loaded at inference time.

**Per-channel training mean**: `[0.9257, 0.9227, 0.9541, 0.9596, 1.0228, 1.0426, 1.0358, 1.0468, 1.1699, 1.1736, 1.0495, 1.0370, 1.2511, 1.6495]`

**Per-channel training std** : `[0.1410, 0.2207, 0.3184, 0.5724, 0.4601, 0.4465, 0.4651, 0.4948, 0.5133, 0.6836, 0.5323, 0.6628, 0.6784, 1.0727]`


In [8]:
from src.detection.preprocessing import NormalizationStats
stats = NormalizationStats.from_json(PROJECT_ROOT / "outputs/detection/data_verification/normalization_statistics.json")
print("mean:", np.round(stats.mean, 3))
print("std :", np.round(stats.std, 3))


mean: [0.926 0.923 0.954 0.96  1.023 1.043 1.036 1.047 1.17  1.174 1.049 1.037
 1.251 1.65 ]
std : [0.141 0.221 0.318 0.572 0.46  0.447 0.465 0.495 0.513 0.684 0.532 0.663
 0.678 1.073]


## SECTION 13 - Train / Validation / Test Handling

The official split is preserved exactly. Filenames are numbered independently
in each split (so `image_1.h5` occurs in all three splits with different
payloads); a per-name overlap check would spuriously report leakage. We keep
the three splits as isolated `Landslide4SenseDataset` objects, each pointing
only to its own directory.

| Split | n_files (img == mask) | Directory |
|-------|---------------------:|-----------|
| train | 3799 | `data/raw/landslide4sense/TrainData/TrainData/img` |
| valid | 245 | `data/raw/landslide4sense/ValidData/ValidData/img` |
| test  | 800  | `data/raw/landslide4sense/TestData/TestData/img` |

See `outputs/detection/data_verification/split_filename_check.json` for the
per-name overlap report and the accompanying explanation.


## SECTION 14 - Class Distribution Analysis

Masks are strictly binary. Value 0 encodes background; value 1 encodes
landslide. There is **no** class-2 or higher anywhere in the dataset.

| Split | Files | Landslide pixel % | Files with any landslide | Per-file mean landslide % | Per-file max landslide % |
|-------|------:|------------------:|-------------------------:|--------------------------:|-------------------------:|
| train | 3799 | 2.318% | 2231 | 2.318% | 47.534% |
| valid | 245 | 1.713% | 146 | 1.713% | 61.652% |
| test  | 800  | 1.889% | 536 | 1.889% | 88.348% |

**Observation:** Strong class imbalance (~2% positive pixels globally,
~40% of patches contain zero landslide pixels). We *record* this here and
defer any remedy (weighted loss, focal loss, sampling) to Stage 2 where the
loss function is chosen. This section is analysis only.


## SECTION 15 - Visualize Raw Data

A single representative training sample (a patch with an obvious landslide
footprint) is used across the visualization sections. The raw feature
values are shown after a 2..98% per-channel display stretch - this is *only*
for visualization; the model receives z-scored values (see Section 25).


![RGB composite (Landslide4Sense B4/B3/B2 = channels 3/2/1), ground-truth mask, and overlay.](../outputs/detection/data_verification/figs/fig02_rgb_mask_overlay.png)

*RGB composite (Landslide4Sense B4/B3/B2 = channels 3/2/1), ground-truth mask, and overlay.*


## SECTION 16 - Visualize All 14 Channels

Every one of the 14 channels for the selected sample:


![All 14 Landslide4Sense channels (per-channel 2..98% display stretch).](../outputs/detection/data_verification/figs/fig01_all14_channels.png)

*All 14 Landslide4Sense channels (per-channel 2..98% display stretch).*


## SECTION 17 - Ground-Truth Mask Verification

For every mask in every split we recorded the set of unique values (see
Section 14). The union across all splits is `{0, 1}`. Mask shape
`(128, 128)` matches image spatial dimensions exactly, so no resampling is
required. Encoding is binary; class 1 = landslide.


## SECTION 18 - Image + Mask Overlay Verification

The overlay above (Section 15) super-imposes mask pixels on the RGB image.
For the chosen sample the mask footprint aligns with a bright scar-like
region visible in the optical channels, which is the expected pattern for a
true landslide detection target.

The DEM and slope channels for the same sample:


![Slope (C12) and DEM (C13) for the same sample.](../outputs/detection/data_verification/figs/fig03_slope_dem.png)

*Slope (C12) and DEM (C13) for the same sample.*


## SECTION 19 - Preprocessing Implementation

All reusable preprocessing lives in `src/detection/preprocessing.py`:

* `read_image(path)` / `read_mask(path)`: HDF5 loaders, shape-checked.
* `sanitize(image)`: replaces NaN / Inf with `0.0` (defensive; not triggered).
* `normalize(image, stats)`: per-channel z-score with train-only stats.
* `to_chw_tensor(image_hwc)`: (H, W, C) numpy -> (C, H, W) float32 tensor.
* `mask_to_tensor(mask_hw)`: uint8 -> float32 {0.0, 1.0} tensor.
* `preprocess_pair(image_path, mask_path, stats)`: the full non-augmentation pipeline.


In [9]:
from src.detection.preprocessing import preprocess_pair, NormalizationStats
stats = NormalizationStats.from_json(PROJECT_ROOT / "outputs/detection/data_verification/normalization_statistics.json")
img_p  = sorted(TRAIN_IMG_DIR.iterdir())[0]
mask_p = sorted(TRAIN_MSK_DIR.iterdir())[0]
x, y = preprocess_pair(img_p, mask_p, stats)
print("image tensor:", tuple(x.shape), x.dtype, f"min={x.min():.3f} max={x.max():.3f}")
print("mask  tensor:", tuple(y.shape), y.dtype, "unique:", sorted(set(y.unique().tolist())))


image tensor: (14, 128, 128) torch.float32 min=-1.844 max=8.226
mask  tensor: (128, 128) torch.float32 unique: [0.0, 1.0]


## SECTION 20 - Dataset Class

`src/detection/dataset.py` defines `Landslide4SenseDataset`. Given an image
directory, a mask directory, `NormalizationStats`, and a split name, it
pairs `image_N.h5` with `mask_N.h5` by numeric id, applies the preprocessing
pipeline, and (for `split='train'` only) applies geometric augmentation.
`__getitem__` returns `(image: (14, 128, 128) float32, mask: (128, 128) float32)`.


In [10]:
from src.detection.dataset import build_all_splits
ds = build_all_splits(DATA_ROOT, stats, augment_seed=SEED)
for name, d in ds.items():
    r = d.report
    print(f"{name:6s} paired={r.n_paired:4d} unpaired_img={len(r.unpaired_image_ids)} "
          f"unpaired_mask={len(r.unpaired_mask_ids)} augment={d._aug is not None}")


train  paired=3799 unpaired_img=0 unpaired_mask=0 augment=True
valid  paired= 245 unpaired_img=0 unpaired_mask=0 augment=False
test   paired= 800 unpaired_img=0 unpaired_mask=0 augment=False


## SECTION 21 - Augmentation Strategy

**Conservative, segmentation-safe, geometric only:**

* Horizontal flip with p = 0.5
* Vertical flip with p = 0.5
* 90-degree rotation, k ~ Uniform{0, 1, 2, 3}

All three transforms are grid-preserving permutations of pixels; they never
interpolate values. This preserves both the physical reflectance values in
channels 0-11 and the DEM/slope values in channels 12-13 exactly.

Every transform is applied with **the same parameters** to the image and to
the mask, so alignment is preserved by construction. Photometric / color
augmentations are deliberately excluded because channels 0-11 encode
reflectance and 12-13 encode terrain - jittering them would violate their
physical meaning.


## SECTION 22 - Augmentation Verification

For a training patch containing a real landslide footprint, the four
augmented (image, mask) pairs still share the same shape and the mask
positive-pixel count is invariant under the augmentation (permutations only
reorder pixels, they don't add or drop them). Alignment is preserved.


![Four random augmentations of the same (image, mask) pair - alignment preserved.](../outputs/detection/data_verification/figs/fig04_augmentation_pairs.png)

*Four random augmentations of the same (image, mask) pair - alignment preserved.*


## SECTION 23 - DataLoader

Each split has its own `DataLoader`. Only the training loader shuffles by
default. The training loader also carries the augmentation; validation and
test loaders do not.


In [11]:
import torch
from src.detection.dataset import build_dataloader
loaders = {name: build_dataloader(d, batch_size=8, num_workers=0) for name, d in ds.items()}
for name, loader in loaders.items():
    xb, yb = next(iter(loader))
    print(f"{name:6s} x={tuple(xb.shape)} y={tuple(yb.shape)} x.dtype={xb.dtype} y.dtype={yb.dtype}")


train  x=(8, 14, 128, 128) y=(8, 128, 128) x.dtype=torch.float32 y.dtype=torch.float32
valid  x=(8, 14, 128, 128) y=(8, 128, 128) x.dtype=torch.float32 y.dtype=torch.float32
test   x=(8, 14, 128, 128) y=(8, 128, 128) x.dtype=torch.float32 y.dtype=torch.float32


## SECTION 24 - Batch Verification

For a randomly-shuffled 64-item training batch, per-channel mean stays near
0 and per-channel std stays near 1 (loose bounds shown; exact values change
batch to batch by design). All tensor values are finite; all mask values
are in `{0.0, 1.0}`.


In [12]:
loader = build_dataloader(ds["train"], batch_size=64, num_workers=0)
xb, yb = next(iter(loader))
assert xb.shape == (64, 14, 128, 128), xb.shape
assert yb.shape == (64, 128, 128), yb.shape
assert xb.dtype == torch.float32 and yb.dtype == torch.float32
assert torch.isfinite(xb).all() and torch.isfinite(yb).all()
assert set(torch.unique(yb).tolist()).issubset({0.0, 1.0})
per_ch_mean = xb.mean(dim=(0,2,3))
per_ch_std  = xb.std (dim=(0,2,3))
print("max|mean| :", float(per_ch_mean.abs().max()))
print("std range :", float(per_ch_std.min()), float(per_ch_std.max()))


max|mean| : 0.08234740793704987
std range : 0.8395318388938904 1.116209864616394


## SECTION 25 - Post-Preprocessing Visual Verification

Raw vs. z-scored view of four representative channels. Spatial structure is
preserved (the normalization is a per-channel affine transform) and no
NaN / Inf is introduced.


![Raw vs. z-scored channels for the sample (C0, C3=Red, C12=Slope, C13=DEM).](../outputs/detection/data_verification/figs/fig05_pre_vs_post_norm.png)

*Raw vs. z-scored channels for the sample (C0, C3=Red, C12=Slope, C13=DEM).*


Per-channel value distributions on a 100-patch subset, raw vs. z-scored:


![Per-channel raw (blue) vs. z-scored (orange) value distributions.](../outputs/detection/data_verification/figs/fig06_channel_distributions.png)

*Per-channel raw (blue) vs. z-scored (orange) value distributions.*


## SECTION 26 - Data Leakage Check

The end-to-end verification script (`verify_pipeline`) enforces these
invariants and its full report is saved to
`outputs/detection/data_verification/stage1_validation_report.txt`.

* **Normalization leakage: PASS** - `computed_on == train_split_only`,
  `n_train_files == 3799` matches the training loader.
* **Split leakage: PASS** - each split has its own directory-scoped dataset
  object; no cross-split file sharing.
* **Augmentation leakage: PASS** - only the training dataset receives an
  augmenter; validation and test datasets have `_aug is None`.
* **Test contamination: PASS** - test set never influenced normalization,
  augmentation, or split composition.


## SECTION 27 - Save Preprocessing Statistics

Files under `outputs/detection/data_verification/` (all values from actual
measurements, none fabricated):

* `channel_statistics.csv` - per-channel min/max/mean/std/nan/inf.
* `training_channel_stats.json` - the same data with pixel counts.
* `mask_class_distribution.json` - per-split class distribution.
* `normalization_statistics.json` - **reusable normalization payload**.
* `dataset_summary.json` - resolved paths, shapes, counts.
* `split_filename_check.json` - per-name overlap explanation.
* `stage1_validation_report.txt` - PASS/FAIL log of the verification run.
* `figs/` - the six PNGs shown above.


## SECTION 28 - Save Configuration

`configs/detection.yaml` is updated with the verified values only:
dataset paths and counts, image/mask shapes and keys, the channel label
table, the normalization statistics-file path, and the augmentation methods.
Training hyperparameters remain commented out until Stage 2.


## SECTION 29 - Final Stage-1 Validation

Output of `verify_pipeline.py` (also saved to
`outputs/detection/data_verification/stage1_validation_report.txt`):


In [13]:
print((PROJECT_ROOT / "outputs/detection/data_verification/stage1_validation_report.txt").read_text())


== Loading normalization stats (train-only) ==
  [PASS] normalization stats loaded  mean.shape=(14,) std.shape=(14,)

== Building datasets ==
  [PASS] train split paired  images=3799 masks=3799 paired=3799
  [PASS] valid split paired  images=245 masks=245 paired=245
  [PASS] test split paired  images=800 masks=800 paired=800

== Batch shape / dtype / finiteness ==
  [PASS] train shape  x=(4, 14, 128, 128) y=(4, 128, 128)
  [PASS] train dtype  x=torch.float32 y=torch.float32
  [PASS] train finite  x_min=-2.066 x_max=+8.461
  [PASS] train mask values in {0,1}  unique=[0.0, 1.0]
  [PASS] valid shape  x=(4, 14, 128, 128) y=(4, 128, 128)
  [PASS] valid dtype  x=torch.float32 y=torch.float32
  [PASS] valid finite  x_min=-2.119 x_max=+18.616
  [PASS] valid mask values in {0,1}  unique=[0.0, 1.0]
  [PASS] test shape  x=(4, 14, 128, 128) y=(4, 128, 128)
  [PASS] test dtype  x=torch.float32 y=torch.float32
  [PASS] test finite  x_min=-2.183 x_max=+44.688
  [PASS] test mask values in {0,1}  uniqu

==================================================
STAGE 1 VALIDATION REPORT
==================================================

| Check | Result |
|-------|:------:|
| Dataset discovery                                             | PASS |
| HDF5 inspection                                               | PASS |
| Image structure  (`img`, (128,128,14), float64)               | PASS |
| Mask structure   (`mask`, (128,128), uint8, values {0,1})     | PASS |
| Channel count = 14                                            | PASS |
| Channel mapping                                               | PASS |
| Statistics (full training set)                                | PASS |
| Invalid values (NaN=0, Inf=0)                                 | PASS |
| Normalization (train-only z-score, persisted)                 | PASS |
| Train/Val/Test split preserved                                | PASS |
| Class distribution measured (binary, ~2% positive)            | PASS |
| Augmentation (geometric, train-only)                          | PASS |
| Dataset class                                                 | PASS |
| DataLoader                                                    | PASS |
| Tensor verification (shape/dtype/finite)                      | PASS |
| Visualization                                                 | PASS |
| Leakage checks                                                | PASS |

**Overall: STAGE 1 READY FOR U-NET TRAINING**


## SECTION 30 - Final Data Preparation Report

1. **Dataset location:** `data/raw/landslide4sense/{Train,Valid,Test}Data/{Train,Valid,Test}Data/{img,mask}/*.h5` (nested layout preserved from the archive).
2. **Files per split:** train 3799 / valid 245 / test 800 (image count == mask count in every split).
3. **Verified image dimensions:** `(128, 128, 14)`, float64.
4. **Verified channel count:** 14.
5. **Channel mapping:** Sentinel-2 bands B1..B12 (channels 0..11) + Slope (12) + DEM (13), per Ghorbanzadeh et al. 2022; see Section 8 and `configs/detection.yaml`.
6. **Mask encoding:** binary `uint8`, `{0 = background, 1 = landslide}`, shape `(128, 128)`.
7. **Normalization:** per-channel z-score, statistics computed on train split only (3799 files, 62,242,816 px / channel), stored in `outputs/detection/data_verification/normalization_statistics.json`.
8. **Invalid value handling:** none required (NaN=0, Inf=0 on training set); defensive `nan_to_num` retained in preprocessing for robustness.
9. **Class distribution:** ~2.32% landslide pixels overall (train 2.32% / valid 1.71% / test 1.89%). Strong imbalance; loss weighting to be chosen in Stage 2.
10. **Augmentation strategy:** geometric only (hflip, vflip, rot90 x k), training loader only, applied identically to image and mask.
11. **DataLoader tensor format:** image `(B, 14, 128, 128) float32`, mask `(B, 128, 128) float32` in {0.0, 1.0}.
12. **Files saved:** see Section 27.
13. **Unresolved uncertainties:** minima of exactly `0.0` on every channel are consistent with a NoData / masked-out convention but cannot be geolocated to specific pixels without accompanying metadata; kept as-is for Stage 1.
14. **Final Stage-1 status:** **COMPLETE - ready for U-Net training** (which is out of scope for this task).
